In [4]:
from datetime import datetime, date
from typing import List, Dict
import uuid

class Usuario:
    def __init__(self, username: str, contrasenya: str, email: str):
        self.username = username
        self.contrasenya = contrasenya
        self.email = email
    
    def registrar(self):
        # Aquí iría la lógica de registro en la base de datos
        pass

class Persona(Usuario):
    def __init__(self, username: str, contrasenya: str, email: str, nombre: str, telefon: str):
        super().__init__(username, contrasenya, email)
        self.nombre = nombre
        self.telefon = telefon
        self.reservas: List[Reserva] = []
    
    def modificar_usuario(self, **kwargs):
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
    
    def crear_reserva_multiple(self, asientos: List[int], funcion: 'Funcion') -> 'Reserva':
        if funcion.verificar_disponibilidad_multiple(asientos):
            reserva = Reserva(asientos, funcion)
            self.reservas.append(reserva)
            funcion.sala.actualizar_asientos(asientos, False)
            return reserva
        raise ValueError("Algunos asientos no están disponibles")

class Sala:
    def __init__(self, aforo: int):
        self.aforo = aforo
        self.disponibilidad = True
        self.asientos = {i: True for i in range(1, aforo + 1)}  # True = disponible
    
    def comprobar_disponibilidad(self) -> bool:
        return any(self.asientos.values())
    
    def get_asientos_disponibles(self) -> List[int]:
        return [num for num, disp in self.asientos.items() if disp]
    
    def actualizar_asientos(self, asientos: List[int], estado: bool):
        for asiento in asientos:
            if asiento in self.asientos:
                self.asientos[asiento] = estado

class Pelicula:
    def __init__(self, titulo: str, sinopsis: str, duracion: int):
        self.titulo = titulo
        self.sinopsis = sinopsis
        self.duracion = duracion
        self.activa = True
    
    def detalles(self) -> Dict:
        return {
            "titulo": self.titulo,
            "sinopsis": self.sinopsis,
            "duracion": self.duracion,
            "activa": self.activa
        }

class Empleado:
    def __init__(self, num_registro: str, puesto: str, fecha_ingreso: str):
        self.num_registro = num_registro
        self.puesto = puesto
        self.fecha_ingreso = fecha_ingreso
    
    def agregar_funcion(self, funcion: 'Funcion') -> bool:
        # Verificar permisos y agregar función
        return True
    
    def agregar_pelicula(self, pelicula: Pelicula) -> bool:
        # Verificar permisos y agregar película
        return True
    
    def agregar_promocion(self, promocion: 'Promocion') -> bool:
        # Verificar permisos y agregar promoción
        return True
    
    def modificar_estado_pelicula(self, pelicula: Pelicula, estado: bool):
        pelicula.activa = estado

class Zona_Comida(Empleado):
    def __init__(self, num_registro: str, puesto: str, fecha_ingreso: str):
        super().__init__(num_registro, puesto, fecha_ingreso)
        self.productos: List[str] = []
        self.precios: Dict[str, float] = {}
    
    def menu(self) -> Dict[str, float]:
        return self.precios

class Promocion:
    def __init__(self, descuento: float, beneficios: str):
        self.descuento = descuento
        self.beneficios = beneficios
        self.activa = True
    
    def aplicar_promocion(self, precio: float) -> float:
        if self.activa:
            return precio * (1 - self.descuento)
        return precio

class Funcion:
    def __init__(self, hora: datetime, fecha: date, precio: float, sala: Sala, pelicula: Pelicula):
        self.hora = hora
        self.fecha = fecha
        self.precio = precio
        self.sala = sala
        self.pelicula = pelicula
        self.disponible = True
    
    def consultar_asiento(self, numero: int) -> bool:
        return self.sala.asientos.get(numero, False)
    
    def verificar_disponibilidad_multiple(self, asientos: List[int]) -> bool:
        return all(self.consultar_asiento(asiento) for asiento in asientos)

class Reserva:
    def __init__(self, num_asientos: List[int], funcion: Funcion):
        self.num_asientos = num_asientos
        self.fecha = date.today()
        self.estado = "pendiente"
        self.funcion = funcion
        self.codigo_reserva = str(uuid.uuid4())
        self.precio_total = self.calcular_precio_total()
    
    def cancelar_reserva(self):
        if self.estado != "cancelada":
            self.estado = "cancelada"
            self.funcion.sala.actualizar_asientos(self.num_asientos, True)
    
    def confirmar_reserva(self):
        if self.estado == "pendiente":
            self.estado = "confirmada"
    
    def calcular_precio_total(self) -> float:
        return len(self.num_asientos) * self.funcion.precio

# Ejemplo de uso del sistema
def ejemplo_uso():
    # Crear sala
    sala = Sala(100)
    
    # Crear película
    pelicula = Pelicula("Matrix", "Película de ciencia ficción", 150)
    
    # Crear función
    funcion = Funcion(
        datetime.now(),
        date.today(),
        10.0,
        sala,
        pelicula
    )
    
    # Crear usuario y realizar reserva múltiple
    usuario = Persona("john_doe", "password123", "john@example.com", "John Doe", "123456789")
    asientos_deseados = [1, 2, 3]  # Reserva múltiple de 3 asientos
    
    try:
        reserva = usuario.crear_reserva_multiple(asientos_deseados, funcion)
        print(f"Reserva creada exitosamente. Código: {reserva.codigo_reserva}")
        print(f"Total a pagar: ${reserva.precio_total}")
    except ValueError as e:
        print(f"Error al crear la reserva: {e}")

In [8]:
def ejemplo_completo_cine():
    print("\n=== SISTEMA DE CINE ===\n")

    # 1. Creación de Salas
    print("1. Creando salas...")
    sala_vip = Sala(50)  # Sala VIP con menos asientos
    sala_normal = Sala(100)  # Sala normal
    sala_premium = Sala(75)  # Sala premium
    
    # 2. Creación de Películas
    print("\n2. Registrando películas...")
    peliculas = [
        Pelicula("Rapidos y Furiosos", "Una familia que adora la velocidad quiere volverse millonaria robandole a la persona que controla Brazil", 148),
        Pelicula("Me before you", "La historia de porque el amor es mejor que todo en esta vida.", 175),
        Pelicula("La purga", "Un grupo de personas sobrevira en la noche donde en USA esta permitido todo delito.", 169)
    ]
    
    # Mostrar detalles de películas
    print("\nDetalles de películas registradas:")
    for pelicula in peliculas:
        detalles = pelicula.detalles()
        print(f"- {detalles['titulo']}: {detalles['duracion']} minutos")
    
    # 3. Creación de Empleados
    print("\n3. Registrando empleados...")
    # Empleado normal
    empleado_general = Empleado("EMP001", "Gerente", "2024-01-15")
    
    # Empleado de zona de comida
    empleado_comida = Zona_Comida("EMP002", "Encargado Comida", "2024-02-01")
    empleado_comida.productos.extend(["Palomitas", "Nachos", "Refrescos", "Hot Dogs"])
    empleado_comida.precios.update({
        "Palomitas": 5.99,
        "Nachos": 6.99,
        "Refrescos": 3.99,
        "Hot Dogs": 4.99
    })
    
    # Mostrar menú
    print("\nMenú de comida disponible:")
    for producto, precio in empleado_comida.menu().items():
        print(f"- {producto}: ${precio}")
    
    # 4. Creación de Funciones
    print("\n4. Programando funciones...")
    funciones = [
        Funcion(datetime.now(), date.today(), 15.0, sala_vip, peliculas[0]),
        Funcion(datetime.now(), date.today(), 10.0, sala_normal, peliculas[1]),
        Funcion(datetime.now(), date.today(), 12.0, sala_premium, peliculas[2])
    ]
    
    # Empleado agrega las funciones
    for funcion in funciones:
        if empleado_general.agregar_funcion(funcion):
            print(f"Función agregada: {funcion.pelicula.titulo} - Sala: {funcion.sala.aforo} asientos")
    
    # 5. Creación de Promociones
    print("\n5. Configurando promociones...")
    promociones = [
        Promocion(0.2, "Descuento de estudiante 20%"),
        Promocion(0.15, "Descuento para adultos mayores 15%"),
        Promocion(0.25, "Promoción 2x1 en miércoles")
    ]
    
    # Empleado agrega las promociones
    for promocion in promociones:
        if empleado_general.agregar_promocion(promocion):
            print(f"Promoción agregada: {promocion.beneficios}")
    
    # 6. Creación de Usuarios y Reservas
    print("\n6. Registrando usuarios y procesando reservas...")
    usuarios = [
        Persona("user1", "pass123", "user1@email.com", "Juan Pérez", "123456789"),
        Persona("user2", "pass456", "user2@email.com", "María García", "987654321")
    ]
    
    # Realizar reservas
    try:
        # Reserva múltiple para primer usuario
        asientos_user1 = [1, 2, 3]  # Reserva familiar
        reserva1 = usuarios[0].crear_reserva_multiple(asientos_user1, funciones[0])
        print(f"\nReserva 1 creada:")
        print(f"- Usuario: {usuarios[0].nombre}")
        print(f"- Película: {funciones[0].pelicula.titulo}")
        print(f"- Asientos: {asientos_user1}")
        print(f"- Código: {reserva1.codigo_reserva}")
        print(f"- Total sin descuento: ${reserva1.precio_total}")
        
        # Aplicar promoción a la primera reserva
        precio_con_descuento = promociones[0].aplicar_promocion(reserva1.precio_total)
        print(f"- Total con descuento estudiante: ${precio_con_descuento}")
        
        # Reserva para segundo usuario
        asientos_user2 = [10, 11]  # Reserva para pareja
        reserva2 = usuarios[1].crear_reserva_multiple(asientos_user2, funciones[1])
        print(f"\nReserva 2 creada:")
        print(f"- Usuario: {usuarios[1].nombre}")
        print(f"- Película: {funciones[1].pelicula.titulo}")
        print(f"- Asientos: {asientos_user2}")
        print(f"- Código: {reserva2.codigo_reserva}")
        print(f"- Total: ${reserva2.precio_total}")
        
    except ValueError as e:
        print(f"Error al crear reserva: {e}")
    
    # 7. Verificación de disponibilidad de asientos
    print("\n7. Verificando disponibilidad de asientos...")
    for funcion in funciones:
        asientos_disponibles = funcion.sala.get_asientos_disponibles()
        print(f"\nSala de {funcion.pelicula.titulo}:")
        print(f"- Asientos disponibles: {len(asientos_disponibles)} de {funcion.sala.aforo}")
    
    # 8. Gestión de reservas
    print("\n8. Gestionando estados de reservas...")
    # Confirmar primera reserva
    reserva1.confirmar_reserva()
    print(f"Reserva {reserva1.codigo_reserva} confirmada")
    
    # Cancelar segunda reserva
    reserva2.cancelar_reserva()
    print(f"Reserva {reserva2.codigo_reserva} cancelada")
    
    # 9. Modificación de usuarios
    print("\n9. Actualizando información de usuario...")
    usuarios[0].modificar_usuario(telefon="2228181976")
    print(f"Teléfono actualizado para {usuarios[0].nombre}: {usuarios[0].telefon}")
    
    # 10. Gestión de películas
    print("\n10. Gestionando estado de películas...")
    # Desactivar una película
    empleado_general.modificar_estado_pelicula(peliculas[1], False)
    print(f"Película '{peliculas[1].titulo}' desactivada")
    
    print("\n=== FIN ===")

if __name__ == "__main__":
    ejemplo_completo_cine()


=== SISTEMA DE CINE ===

1. Creando salas...

2. Registrando películas...

Detalles de películas registradas:
- Rapidos y Furiosos: 148 minutos
- Me before you: 175 minutos
- La purga: 169 minutos

3. Registrando empleados...

Menú de comida disponible:
- Palomitas: $5.99
- Nachos: $6.99
- Refrescos: $3.99
- Hot Dogs: $4.99

4. Programando funciones...
Función agregada: Rapidos y Furiosos - Sala: 50 asientos
Función agregada: Me before you - Sala: 100 asientos
Función agregada: La purga - Sala: 75 asientos

5. Configurando promociones...
Promoción agregada: Descuento de estudiante 20%
Promoción agregada: Descuento para adultos mayores 15%
Promoción agregada: Promoción 2x1 en miércoles

6. Registrando usuarios y procesando reservas...

Reserva 1 creada:
- Usuario: Juan Pérez
- Película: Rapidos y Furiosos
- Asientos: [1, 2, 3]
- Código: eefb6fda-2ba8-4a70-b48e-7b1662d978bf
- Total sin descuento: $45.0
- Total con descuento estudiante: $36.0

Reserva 2 creada:
- Usuario: María García
- 